Extended Features cleaning

In [ ]:
import re
import sys
import os
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from dotenv import load_dotenv
from pathlib import Path

PATH_SEP = os.sep
# Merging the path

sys.path.append(os.path.abspath('../../'))

# 1. Load the .env file
load_dotenv()

folder_path = os.getenv("EXTERNAL_DATA_FOLDER")
dir_files=os.listdir(folder_path)

# Creating File Paths
files=[]
for file in dir_files:
    files.append(folder_path+PATH_SEP+file)

files=sorted(files)
# Reading Data
df=pd.read_csv(files[0])


Information about the joined features dataset

In [ ]:
df.info()

# Handling Nulls

Calculate the null percentage

In [ ]:
null_percentage = (df.isnull().sum() / df.shape[0]) * 100

null_df = null_percentage.reset_index()
null_df.columns = ['Column', 'Null %']

pd.set_option('display.max_rows', None)

print(null_df)

dropped values that has null values >60%



In [ ]:
df.drop(columns=[
    'mid_price_log',
    'cc_contractor_sentiment_overall_score',
    'sentiment_delta'
], inplace=True)

Fill the median value for missing numerical values

In [ ]:

cols_to_impute = [
    'crm_contractor_sentiment_score',
    'crm_agent_chase_count',
    'Time_to_Renewal',
    'crm_delays_in_accreditation',
    'crm_contractor_engagement',
    'crm_customer_payment_intention',
    'crm_competitors_mentioned',
    'crm_membership_overdue',
    'crm_customer_complained',
    'crm_dissatisfaction_with_support'
]

for col in cols_to_impute:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
null_percentage = (df.isnull().sum() / df.shape[0]) * 100

null_df = null_percentage.reset_index()
null_df.columns = ['Column', 'Null %']

pd.set_option('display.max_rows', None)

print(null_df)

# **Scaling the features**

In [ ]:
df.info()

Find the min and max values in the rows

In [ ]:
min_max = df.describe().T[['min', 'max']]

# show all rows
pd.set_option('display.max_rows', None)

print(min_max)

Standard scaling

In [ ]:
from sklearn.preprocessing import StandardScaler


scale_cols = [
    'log_calls','log_days',
    'Total_Renewal_Score_New','Tenure_Years',
    'tenure_days','days_last_call_to_renewal',
    'Total_Calls_In_Window','days_since_last_call',
    'crm_contractor_sentiment_score','call_freq_per_month'
]


scaler_main = StandardScaler()
df[scale_cols] = scaler_main.fit_transform(df[scale_cols])


Handle the infinity and fill the nulls created by handling infinity for price_change_ratio

In [ ]:
import numpy as np

df['price_change_ratio'] = df['price_change_ratio'].replace([np.inf, -np.inf], np.nan)
df['price_change_ratio'].fillna(df['price_change_ratio'].median(), inplace=True)

In [ ]:
# Reduce skew (log)
df['price_change_ratio'] = np.sign(df['price_change_ratio']) * np.log1p(np.abs(df['price_change_ratio']))

scaler_pcr = StandardScaler()
df['price_change_ratio'] = scaler_pcr.fit_transform(df[['price_change_ratio']])

In [ ]:
min_max = df.describe().T[['min', 'max']]

# show all rows
pd.set_option('display.max_rows', None)

print(min_max)

Check whether the data is scaled and normalized

In [ ]:
check = pd.DataFrame({
    'mean': df.mean(),
    'std': df.std(),
    'min': df.min(),
    'max': df.max()
})

print(check)

In [ ]:

output_path_folder = os.getenv("EXTERNAL_DATA_FOLDER")
df.to_csv(output_path_folder+PATH_SEP+"external_cleaned.csv",index=False)